# 🎵 ComposerMind OpenAudio-48k: EDA & PyTorch Dataloader

**Creator & Author**: Boobalan Arjunan  
**Publisher**: ComposerMind Labs  
**License**: Creative Commons Attribution 4.0 International (CC BY 4.0)  
**Audio Specifications**: 48,000 Hz, 24-bit Linear PCM Stereo  
**Total Unique Tracks**: 94 Tracks | 3.88 Audio Hours (13,985.6 seconds) | 3.75 GB  

---

### Overview
Welcome to the official starter notebook for **ComposerMind OpenAudio-48k**, a studio-grade audio benchmark corpus curated specifically for machine learning, neural audio codecs (EnCodec, DAC), and generative music AI (MusicGen, AudioCraft, Stable Audio Open).

This notebook provides:
1. **Exploratory Data Analysis (EDA)** of ground-truth acoustic parameters (BPM, Keys, Camelot Codes, Dynamic Range).
2. **Audio Visualization**: Waveforms and Mel-Spectrograms.
3. **Text-to-Music Training Prompts**: Turnkey captions for generative modeling.
4. **Production PyTorch `Dataset` & `DataLoader`**: Ingestion-ready pipeline for training models.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
import IPython.display as ipd

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Libraries successfully imported!")

## 📁 1. Load Dataset Manifest (`metadata.csv`)
Let's load the comprehensive manifest containing all 27 extracted attributes for the 94 unique tracks.

In [ ]:
# Locate dataset directory
dataset_dir = "../input/composermind-openaudio-48k" if os.path.exists("../input/composermind-openaudio-48k") else "."
csv_path = os.path.join(dataset_dir, "metadata.csv")

df = pd.read_csv(csv_path)
print(f"Loaded manifest with {len(df)} tracks and {len(df.columns)} columns.\n")
df[["file_name", "title", "tempo_bpm", "musical_key", "camelot_code", "harmonic_ratio", "crest_factor_db"]].head(5)

## 📊 2. Acoustic Intelligence & Distribution Analysis (BeatMind AI)
Let's visualize the core acoustic features extracted by the proprietary **BeatMind AI / TempoMind** engine.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Tempo (BPM) Distribution
sns.histplot(df["tempo_bpm"], kde=True, ax=axes[0, 0], color="#3b82f6", bins=18)
axes[0, 0].set_title("Tempo (BPM) Distribution [Avg: 123.8 BPM]", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("BPM")
axes[0, 0].set_ylabel("Track Count")

# 2. Key Distribution
key_counts = df["musical_key"].value_counts().head(8)
sns.barplot(x=key_counts.values, y=key_counts.index, ax=axes[0, 1], palette="viridis")
axes[0, 1].set_title("Top Musical Keys & Scales", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Track Count")

# 3. Dynamic Range (Crest Factor)
sns.histplot(df["crest_factor_db"], kde=True, ax=axes[1, 0], color="#10b981", bins=15)
axes[1, 0].set_title("Dynamic Range / Crest Factor (dB) [Avg: 15.10 dB]", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Crest Factor (dB)")
axes[1, 0].set_ylabel("Track Count")

# 4. Harmonic vs. Percussive Energy Ratio
sns.scatterplot(
    data=df,
    x="harmonic_ratio",
    y="percussive_ratio",
    hue=df["musical_key"].apply(lambda k: "Minor" if "Minor" in str(k) else "Major"),
    palette={"Minor": "#ef4444", "Major": "#8b5cf6"},
    s=80,
    alpha=0.85,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Harmonic vs. Percussive Energy Ratio (%)", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Harmonic Energy (%)")
axes[1, 1].set_ylabel("Percussive Energy (%)")

plt.tight_layout()
plt.show()

## 🎧 3. Audio Waveform & Mel-Spectrogram Inspection
Let's inspect a sample audio track (`OpenAudio48k_Track01.wav`) to examine the waveform dynamics and frequency spectrum.

In [ ]:
# Load first track
sample_track = df.iloc[0]
audio_path = os.path.join(dataset_dir, sample_track["file_name"])

if os.path.exists(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    print(f"Loaded: {sample_track['title']}")
    print(f"Tensor Shape: {waveform.shape} | Sample Rate: {sample_rate} Hz | Bit Depth: {sample_track['bit_depth']}-bit")

    # Plot Waveform
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
    time_axis = torch.linspace(0, waveform.shape[1] / sample_rate, waveform.shape[1])
    ax1.plot(time_axis[::100], waveform[0, ::100].numpy(), color="#2563eb", alpha=0.8)
    ax1.set_title(f"Waveform: {sample_track['title']} ({sample_track['tempo_bpm']} BPM - {sample_track['musical_key']})", fontweight="bold")
    ax1.set_ylabel("Amplitude")
    ax1.set_xlim(0, waveform.shape[1] / sample_rate)

    # Compute & Plot Mel-Spectrogram
    mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=sample_rate, n_mels=80)
    mel_spec = mel_transform(waveform[0:1, :sample_rate * 30]) # First 30s
    mel_db = torchaudio.transforms.AmplitudeToDB()(mel_spec)
    img = ax2.imshow(mel_db[0].numpy(), cmap="magma", origin="lower", aspect="auto")
    ax2.set_title("Mel-Spectrogram (First 30 Seconds)", fontweight="bold")
    ax2.set_xlabel("Frames")
    ax2.set_ylabel("Mel Frequency Bins")
    plt.colorbar(img, ax=ax2, format="%+2.0f dB")

    plt.tight_layout()
    plt.show()
else:
    print(f"Audio file {audio_path} not found in local path.")

## 🤖 4. Text-to-Music Training Prompts
Each track includes a turnkey natural language prompt for training or fine-tuning models like **MusicGen**, **AudioCraft**, or **Stable Audio Open**.

In [ ]:
for idx in [0, 45, 93]:
    row = df.iloc[idx]
    print(f"=== {row['title']} [{row['tempo_bpm']} BPM | {row['musical_key']}] ===")
    print(f"Prompt: \"{row['text_prompt']}\"\n")

## ⚡ 5. Production PyTorch `Dataset` & `DataLoader`
Here is a clean PyTorch `Dataset` implementation that you can drop directly into your neural network training scripts.

In [ ]:
class ComposerMindOpenAudioDataset(Dataset):
    def __init__(self, root_dir, manifest_path=None, target_sample_rate=48000, max_duration_sec=30.0):
        self.root_dir = root_dir
        csv_file = manifest_path or os.path.join(root_dir, "metadata.csv")
        self.df = pd.read_csv(csv_file)
        self.target_sr = target_sample_rate
        self.target_length = int(target_sample_rate * max_duration_sec)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.root_dir, row["file_name"])

        waveform, sr = torchaudio.load(audio_path)

        # Resample if needed
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform = resampler(waveform)

        # Truncate or pad to target length
        channels, length = waveform.shape
        if length > self.target_length:
            waveform = waveform[:, :self.target_length]
        elif length < self.target_length:
            padding = self.target_length - length
            waveform = torch.nn.functional.pad(waveform, (0, padding))

        item = {
            "waveform": waveform,
            "file_name": row["file_name"],
            "tempo_bpm": torch.tensor(row["tempo_bpm"], dtype=torch.float32),
            "musical_key": row["musical_key"],
            "camelot_code": row["camelot_code"],
            "harmonic_ratio": torch.tensor(row["harmonic_ratio"], dtype=torch.float32),
            "crest_factor_db": torch.tensor(row["crest_factor_db"], dtype=torch.float32),
            "text_prompt": row["text_prompt"]
        }
        return item

# Instantiate Dataset and DataLoader
dataset = ComposerMindOpenAudioDataset(dataset_dir, max_duration_sec=10.0)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Successfully initialized PyTorch DataLoader with {len(dataset)} items!")
batch = next(iter(loader))
print("Sample Batch Waveform Tensor Shape:", batch["waveform"].shape)
print("Sample Batch BPM Tensor:", batch["tempo_bpm"])
print("Sample Batch Keys:", batch["musical_key"])

## 📜 Citation & Attribution
If you use this dataset in your research, model benchmarking, or creative projects, please cite:

```bibtex
@dataset{arjunan2024composermindopenaudio,
  title     = {ComposerMind OpenAudio-48k: A High-Fidelity Audio Dataset for Machine Learning Research},
  author    = {Arjunan, Boobalan},
  publisher = {ComposerMind Labs},
  year      = {2024},
  license   = {Creative Commons Attribution 4.0 International (CC BY 4.0)}
}
```

**Published by ComposerMind Labs | All Rights Reserved under CC BY 4.0.**